1.  What is Generative AI and what are its primary use cases across
industries?

Ans- Generative AI

**Function**: Uses neural networks to create original content (text, code, images, audio) by identifying patterns in training data. Powered by Large Language Models (LLMs) and Diffusion Models.

Industry Use Cases

**Finance**: Automating reports, risk simulations, and fraud pattern generation.

**Tech**: Automated coding, debugging, and synthetic data generation.

**Healthcare**: Accelerating drug discovery and summarizing clinical notes.

**Marketing**: Generating personalized ad copy, graphics, and social media content.

**Manufacturing**: Generative design for parts and predictive maintenance simulations.

**Retail**: 3D product modeling and advanced AI customer assistants.

2. Explain the role of probabilistic modeling in generative models. How do
these models differ from discriminative models?

Ans- **Probabilistic Modeling in Generative Models**

Joint Distribution: Learns $P(X, Y)$ to understand the underlying structure of the data.

Sampling: Enables the model to "sample" from learned distributions to create new, original data points.

Stochasticity: Introduces randomness so the model generates diverse outputs rather than the same result every time.

**Generative vs. Discriminative Models**

Objective: Generative models learn the data's distribution; Discriminative models learn the boundary between classes.

Math: Generative models $P(X, Y)$; Discriminative models $P(Y | X)$.

Output: Generative creates new instances; Discriminative predicts labels for existing instances.

Example: Generative is learning what a "cat" looks like to draw one; Discriminative is learning the difference between "cat" and "dog" to label a photo.

3. What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?

Ans- **Latent Space**: AEs map text to a fixed point (deterministic); VAEs map text to a distribution (probabilistic).

**Continuity**: AE latent space is discontinuous (gaps lead to gibberish); VAE latent space is continuous (smooth transitions between sentences).

**Generation**: AEs are for reconstruction/compression; VAEs are for sampling/creating new, diverse text.

**Output**: Sampling from an AE usually produces nonsense; sampling from a VAE produces coherent variations.

**Math**: VAEs use KL-Divergence to regularize the latent space; AEs do not.

4. Describe the working of attention mechanisms in Neural Machine
Translation (NMT). Why are they critical?

Ans- **Attention Works in NMT**

Alignment Scoring: The model calculates similarity between the current decoder state and all encoder hidden states.

Weighting: Scores are converted into probabilities (0 to 1) via Softmax, identifying which source words are most relevant.

Context Vector: A weighted average of all encoder states is created, providing a focused "snapshot" of the source sentence for the current translation step.

Dynamic Focus: This process repeats for every single word generated, shifting focus across the source text.

**Why It Is Critical**

Eliminates Bottlenecks: Prevents the loss of information caused by compressing long sentences into a single fixed-length vector.

Long-Range Dependencies: Connects related words regardless of how far apart they are in a sentence (vital for different language structures).

Better Gradients: Provides shorter paths for backpropagation, reducing the vanishing gradient problem.

Explainability: Allows humans to visualize "Attention Maps" to see exactly which source words triggered a specific translation.

5. What ethical considerations must be addressed when using generative AI
for creative content such as poetry or storytelling?

Ans- Copyright & Ownership: Unclear legal standing on whether AI-generated poems/stories can be copyrighted or if they infringe on training data artists.

Plagiarism: Risk of "echoing" existing works or mimicking a specific author's unique voice without compensation or credit.

Bias & Stereotypes: Models often replicate cultural, gender, or racial biases found in their training datasets.

Devaluation of Labor: Massive AI output can saturate markets, making it harder for human authors to earn a living.

Transparency: The moral obligation to disclose if a work was human-written or machine-generated.

Homogenization: AI tends to produce "average" content, potentially stifling radical creative innovation and diverse perspectives.

6. Use the following small text dataset to train a simple Variational
Autoencoder (VAE) for text reconstruction:

["The sky is blue", "The sun is bright", "The grass is green",
"The night is dark", "The stars are shining"]

Preprocess the data (tokenize and pad the sequences).
Build a basic VAE model for text reconstruction.
Train the model and show how it reconstructs or generates similar sentences.


In [4]:
import numpy as np
import tensorflow as tf
import keras
from keras import layers, models, losses, ops

data = ["The sky is blue", "The sun is bright", "The grass is green",
        "The night is dark", "The stars are shining"]

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(data)
sequences = tokenizer.texts_to_sequences(data)
vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(s) for s in sequences)
X = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, padding='post')

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = ops.shape(z_mean)[0]
        dim = ops.shape(z_mean)[1]
        # Use keras.random.normal instead of ops.random_normal
        epsilon = keras.random.normal(shape=(batch, dim))
        return z_mean + ops.exp(0.5 * z_log_var) * epsilon

class VAE(models.Model):
    def __init__(self, vocab_size, max_len, latent_dim=2, emb_dim=8):
        super().__init__()
        self.sampling = Sampling()
        self.encoder_input = layers.Input(shape=(max_len,))
        x = layers.Embedding(vocab_size, emb_dim)(self.encoder_input)
        x = layers.LSTM(16)(x)
        z_m = layers.Dense(latent_dim)(x)
        z_v = layers.Dense(latent_dim)(x)
        self.encoder = models.Model(self.encoder_input, [z_m, z_v])

        self.decoder_input = layers.Input(shape=(latent_dim,))
        x = layers.RepeatVector(max_len)(self.decoder_input)
        x = layers.LSTM(16, return_sequences=True)(x)
        out = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)
        self.decoder = models.Model(self.decoder_input, out)

        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encoder(data)
            z = self.sampling([z_mean, z_log_var])
            reconstruction = self.decoder(z)
            reconstruction_loss = ops.mean(ops.sum(losses.sparse_categorical_crossentropy(data, reconstruction), axis=-1))
            kl_loss = -0.5 * ops.mean(1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        return {"loss": self.total_loss_tracker.result()}

vae = VAE(vocab_size, max_len)
vae.compile(optimizer='adam')
vae.fit(X, epochs=500, verbose=0)

idx_to_word = {i: w for w, i in tokenizer.word_index.items()}
def decode(seq):
    return " ".join([idx_to_word.get(int(i), "") for i in seq if i > 0])

z_mean, _ = vae.encoder.predict(X, verbose=0)
preds = np.argmax(vae.decoder.predict(z_mean, verbose=0), axis=-1)
for i in range(len(data)):
    print(f"Orig: {data[i]} | Rec: {decode(preds[i])}")

Orig: The sky is blue | Rec: the is is blue
Orig: The sun is bright | Rec: the is is bright
Orig: The grass is green | Rec: the is is blue
Orig: The night is dark | Rec: the is is bright
Orig: The stars are shining | Rec: the are are shining


**Explanation of the VAE Logic**

Preprocessing: Maps words to unique integers and pads sentences to a fixed length (L=4) for neural network compatibility.

Sampling Layer: Instead of a fixed vector, the model outputs a mean and variance. It uses the "reparameterization trick" to allow gradient descent despite the random sampling.

Encoder: Compresses the text sequence into a 2D latent space, grouping semantically similar sentences together.

Decoder: Takes a 2D point and reconstructs the original word sequence by expanding the vector back across time steps.

Loss Function: * Reconstruction Loss: Ensures the generated text matches the input.

KL Divergence: Regularizes the latent space into a smooth, standard normal distribution to allow for meaningful generation.

7. Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short
English paragraph into French and German. Provide the original and translated text.

In [10]:
from transformers import MarianMTModel, MarianTokenizer

def translate(text, model_name):
    # Load tokenizer and model directly
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)

    # Tokenize and generate
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = model.generate(**inputs)

    # Decode result
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Define models
fr_model = "Helsinki-NLP/opus-mt-en-fr"
de_model = "Helsinki-NLP/opus-mt-en-de"

text = "The quick brown fox jumps over the lazy dog. It is a beautiful day for learning AI."

# Execute
print(f"Original: {text}")
print("-" * 30)
print(f"French: {translate(text, fr_model)}")
print(f"German: {translate(text, de_model)}")

Original: The quick brown fox jumps over the lazy dog. It is a beautiful day for learning AI.
------------------------------


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

German: Der schnelle Braunfuchs springt über den faulen Hund. Es ist ein schöner Tag, um KI zu lernen.


Original Text: The quick brown fox jumps over the lazy dog. It is a beautiful day for learning AI.

French: Le renard brun rapide saute sur le chien paresseux. C'est une belle journée pour apprendre l'IA.

German: Der schnelle Braunfuchs springt über den faulen Hund. Es ist ein schöner Tag, um KI zu lernen.

8. Implement a simple attention-based encoder-decoder model for
English-to-Spanish translation using Tensorflow or PyTorch.

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim

english_sentences = ["the sun is hot", "the sky is blue", "the grass is green"]
spanish_sentences = ["el sol esta caliente", "el cielo es azul", "la hierba es verde"]

def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
    for s in sentences:
        for word in s.split():
            if word not in vocab: vocab[word] = len(vocab)
    return vocab

en_vocab = build_vocab(english_sentences)
es_vocab = build_vocab(spanish_sentences)
rev_es_vocab = {v: k for k, v in es_vocab.items()}

def tokenize(s, vocab):
    return [vocab["<SOS>"]] + [vocab[w] for w in s.split()] + [vocab["<EOS>"]]

X = [torch.tensor(tokenize(s, en_vocab)) for s in english_sentences]
Y = [torch.tensor(tokenize(s, es_vocab)) for s in spanish_sentences]

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src).unsqueeze(0)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0).unsqueeze(0)
        embedded = self.embedding(input)

        attn_weights = torch.softmax(torch.bmm(hidden.transpose(0, 1), encoder_outputs.transpose(1, 2)), dim=-1)
        context = torch.bmm(attn_weights, encoder_outputs)

        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden

hid_dim = 32
encoder = Encoder(len(en_vocab), 16, hid_dim)
decoder = AttentionDecoder(len(es_vocab), 16, hid_dim)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()))
criterion = nn.CrossEntropyLoss()

for epoch in range(100):
    total_loss = 0
    for src, trg in zip(X, Y):
        optimizer.zero_grad()
        enc_outputs, hidden = encoder(src)
        input = trg[0]
        loss = 0
        for t in range(1, len(trg)):
            output, hidden = decoder(input, hidden, enc_outputs)
            loss += criterion(output, trg[t].unsqueeze(0))
            input = trg[t]
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

def evaluate(sentence):
    with torch.no_grad():
        src = torch.tensor(tokenize(sentence, en_vocab))
        enc_outputs, hidden = encoder(src)
        input = torch.tensor(es_vocab["<SOS>"])
        result = []
        for _ in range(10):
            output, hidden = decoder(input, hidden, enc_outputs)
            pred = output.argmax(1).item()
            if pred == es_vocab["<EOS>"]: break
            result.append(rev_es_vocab[pred])
            input = torch.tensor(pred)
        return " ".join(result)

for s in english_sentences:
    print(f"{s} -> {evaluate(s)}")

the sun is hot -> el sol esta caliente
the sky is blue -> el cielo es azul
the grass is green -> la hierba es verde


9. Use the following short poetry dataset to simulate poem generation with a
pre-trained GPT model:

["Roses are red, violets are blue,",

"Sugar is sweet, and so are you.",

"The moon glows bright in silent skies,",

"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns.
Include your code, the prompt used, and the generated poem in your answer.

In [12]:
from transformers import pipeline

# Load lightweight GPT-2 (approx 500MB)
generator = pipeline('text-generation', model='gpt2')

# Define the reference dataset
dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]

# Create the prompt by joining the dataset with newlines
# We add a leading phrase to trigger a new poem
prompt = "\n".join(dataset) + "\nGold leaves fall on the forest floor,"

# Generate the completion
# max_new_tokens=30 keeps it short (2-4 lines)
result = generator(prompt, max_new_tokens=30, temperature=0.8, num_return_sequences=1, truncation=True)

# Output results
print("--- Generated Poem ---")
print(result[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Generated Poem ---
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.
Gold leaves fall on the forest floor,
And water flows through the black veins.
And the land is covered with black trees.
There are many things about your life that you wish


**Prompt used**

Roses are red, violets are blue,
Sugar is sweet, and so are you.

The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

**Generated Poem**

Roses are red, violets are blue,
Sugar is sweet, and so are you.

The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

Gold leaves fall on the forest floor,
And water flows through the black veins.

And the land is covered with black trees.

There are many things about your life that you wish

10. Imagine you are building a creative writing assistant for a publishing
company. The assistant should generate story plots and character descriptions usingGenerative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challengesyou might face.


Ans- **Model Selection:**

Architecture: Use a Large Language Model (LLM) like GPT-4 or Claude 3 for high-level plot reasoning.

Specialization: Implement LoRA (Low-Rank Adaptation) to fine-tune on a proprietary corpus of high-quality literature for stylistic consistency.

Retrieval: Use a Vector Database (e.g., Pinecone) to store genre-specific tropes and lore for "Augmented Generation" (RAG).

**Training Data:**

Primary Source: Licensed datasets of classic and contemporary novels, screenplays, and character biographies.

Augmentation: Synthetic data generated by models to fill gaps in niche genres or specific cultural contexts.

**Bias Mitigation:**

Adversarial Testing: Use "Red Teaming" to identify and patch stereotypical character archetypes (e.g., gendered roles).

Diversity Constraints: Integrate a "Diversity Controller" in the prompt layer to force varied character backgrounds and traits.

Filtering: Use a Toxicity Classifier to strip harmful or offensive content during the generation phase.

**Evaluation Methods:**

Perplexity & BLEU: For technical linguistic accuracy.

Human-in-the-loop (HITL): Professional editors rate plots for "narrative tension" and "character depth."

Novelty Scoring: Compare generated plots against a database of existing works to ensure originality.

**Real-World Challenges:**

Long-Term Coherence: Maintaining consistency in character traits and plot points across a full-length novel.

Legal/Copyright: Navigating the "Fair Use" of training data and establishing ownership of AI-assisted manuscripts.

Creativity Bottleneck: Preventing "Genericism," where the AI defaults to predictable clichés rather than unique storytelling.